# Thai Legal RAG — Clean Experimental Notebook

เปรียบเทียบ Binary BoW, TF-IDF, Word2Vec Skip-gram และ BGE-M3 dense สำหรับค้นมาตรากฎหมายไทย แล้วตรวจผลต่อคำตอบ โดย 1 มาตราเป็น 1 หน่วยค้นคืน

**วิธีใช้:** เปิดใน Colab → เลือก GPU (จำเป็นในทางปฏิบัติสำหรับ BGE และ generator) → Run all ในโหมด pilot ก่อน → ตรวจ audit และผล → ล็อกค่าและ revision จาก dev ก่อนเปลี่ยนเป็น test. Generator อาจใช้เวลานานมาก ปรับ `RUN_GENERATION` ได้. ผลอยู่ใน `rag_results/` ซึ่งเป็นพื้นที่ชั่วคราว: ดาวน์โหลด ZIP ก่อน runtime หมดอายุ.

งานนี้เป็น benchmark กฎหมายตาม snapshot ไม่ใช่คำปรึกษากฎหมายหรือการยืนยันกฎหมายฉบับปัจจุบัน. ถ้าเคยใช้ผล test เลือกค่ามาก่อน ต้องเปิดเผยข้อจำกัดนั้น. Notebook นี้มีผลลัพธ์เริ่มต้นว่าง ต้องรันบน Colab เพื่อได้ผลจริง.

## 1. Setup และ configuration
ตั้งค่า `MODE="pilot"` เพื่อพัฒนา ใช้ `MODE="test"` หลังล็อกการตั้งค่า. ชุด test ที่จับคู่ gold ไม่ครบจะถูกกันออกจากคะแนนหลักและบันทึกเป็น audit; รายงานความครอบคลุมเสมอ. `RUN_GENERATION=False` ช่วยรันเฉพาะ retrieval.

In [ ]:
%pip install -q "datasets>=3.2,<5" "pythainlp>=5,<6" "gensim>=4.4,<5" "transformers>=4.51,<5" "sentence-transformers>=4.1,<6" "torch>=2.6"

In [ ]:
import gc
import random
import os
import platform
import json
import re
import unicodedata
from collections import defaultdict
from datetime import datetime
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from gensim.models import Word2Vec
from huggingface_hub import HfApi
from IPython.display import display
from pythainlp.tokenize import word_tokenize
from scipy.sparse import issparse
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

RUN_GENERATION = True
MODE = "pilot"                 # "pilot" หรือ "test"
PILOT_QUESTIONS = 40
PILOT_GENERATION_QUESTIONS = 4  # เปลี่ยนเป็น 40 เพื่อประเมิน pilot เต็ม
TEST_GENERATION_QUESTIONS = 200
SEED = 42

TOP_K, RANK_K = 5, 10
MAX_INPUT_TOKENS = 4096        # นับทั้งคำถาม คำสั่ง ตัวบท และ chat template
MAX_NEW_TOKENS = 256
BGE_MAX_TOKENS, BGE_BATCH_SIZE = 4096, 8
BGE_ID = "BAAI/bge-m3"
GENERATOR_ID = "typhoon-ai/llama3.2-typhoon2-1b-instruct"
DATASETS = {
    "corpus": ("VISAI-AI/nitibench-statute", "ccl"),
    "pilot": ("airesearch/WangchanX-Legal-ThaiCCL-RAG", "train"),
    "test": ("VISAI-AI/nitibench", "ccl"),
}

if MODE not in {"pilot", "test"}:
    raise ValueError('MODE ต้องเป็น "pilot" หรือ "test"')
if min(PILOT_QUESTIONS, PILOT_GENERATION_QUESTIONS, TEST_GENERATION_QUESTIONS) < 1:
    raise ValueError("จำนวนคำถามต้องมากกว่า 0")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
RUN_DIR = Path("rag_results") / datetime.now().strftime(f"{MODE}_%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True)
print(f"Mode: {MODE} | Device: {DEVICE} | Results: {RUN_DIR}")

## 2. โหลดข้อมูลและบันทึกรุ่น
หนึ่งแถวของ corpus ใช้เป็นหนึ่งมาตรา; เก็บตัวบทต้นฉบับแยกจากข้อความที่ normalize. Revision จะถูก resolve ณ เวลารันและบันทึกไว้ เพื่อทำซ้ำควรนำ SHA ไป pin ในรอบ final.

In [ ]:
api = HfApi()
dataset_revisions, raw = {}, {}
for name, (repo, split) in DATASETS.items():
    dataset_revisions[name] = api.dataset_info(repo).sha
    raw[name] = load_dataset(repo, split=split, revision=dataset_revisions[name])
    print(name, len(raw[name]), raw[name].column_names, dataset_revisions[name])

required = {"corpus": {"law_name", "section_num", "section_content"},
            "pilot": {"question", "positive_contexts", "positive_answer"},
            "test": {"question", "relevant_laws", "reference_answer"}}
for name, cols in required.items():
    missing = cols - set(raw[name].column_names)
    if missing: raise ValueError(f"{name}: missing columns {sorted(missing)}")

manifest = {"mode": MODE, "seed": SEED, "datasets": DATASETS,
            "dataset_revisions": dataset_revisions, "python": platform.python_version(),
            "versions": {p: version(p) for p in ["datasets", "pythainlp", "gensim", "scikit-learn", "sentence-transformers", "transformers", "torch"]}}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. Corpus และการจับคู่ gold
เก็บเลขมาตราเป็นข้อความ; normalize เลขไทยและช่องว่างเฉพาะเพื่อ matching/retrieval. `Sxxxxx` เป็น ID ภายใน snapshot ไม่ใช่รหัสมาตราสากล: เก็บ mapping เสมอ. ไม่มีการทิ้งรายการซ้ำอัตโนมัติ. Gold ทุก entry ต้องจับคู่ได้แน่นอนจึงเข้า primary evaluation; รายละเอียดที่จับไม่ได้ส่งออกแยก. ตรวจ duplicate query ข้ามชุดเฉพาะ exact normalized match; near-duplicate ยังต้องตรวจเพิ่มเติม.

In [ ]:
THAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
def search_text(text):
    text = unicodedata.normalize("NFC", str(text) if text is not None else "").translate(THAI_DIGITS)
    return " ".join(text.replace("\u200b", " ").replace("\ufeff", "").split())

def law_key(name, section):
    name = "".join(search_text(name).split())
    section = "".join(search_text(section).split()).removeprefix("มาตรา")
    return name, section

corpus, statute_lookup = {}, defaultdict(list)
for index, row in enumerate(raw["corpus"], 1):
    key = law_key(row.get("law_name"), row.get("section_num"))
    if not all(key) or not search_text(row.get("section_content")):
        raise ValueError(f"corpus row {index}: missing law, section, or text")
    sid = f"S{index:05d}"
    corpus[sid] = dict(row)
    statute_lookup[key].append(sid)

duplicate_keys = [{"law_key": k[0], "section_key": k[1], "ids": v} for k,v in statute_lookup.items() if len(v)>1]
print("Corpus:", len(corpus), "| normalized duplicate keys:", len(duplicate_keys))
pd.DataFrame(duplicate_keys, columns=["law_key", "section_key", "ids"]).to_json(RUN_DIR / "duplicate_keys.jsonl", orient="records", lines=True, force_ascii=False)
pd.DataFrame([{"statute_id": sid, "law_name": row.get("law_name"), "section_num": row.get("section_num"), "law_code": row.get("law_code")} for sid,row in corpus.items()]).to_csv(RUN_DIR / "statute_ids.csv", index=False, encoding="utf-8-sig")
display(pd.DataFrame([{"statute_id": sid, **row} for sid,row in list(corpus.items())[:2]]))

In [ ]:
def match_statute(name, section, text):
    key = law_key(name, section)
    candidates = statute_lookup.get(key, []) if all(key) else []
    if len(candidates) == 1: return candidates[0], "matched_key"
    if len(candidates) > 1 and search_text(text):
        exact = [sid for sid in candidates if search_text(corpus[sid]["section_content"]) == search_text(text)]
        if len(exact) == 1: return exact[0], "matched_text"
    return None, "ambiguous" if candidates else "unmatched"

def gold_entries(row, split):
    if split == "pilot":
        for item in row.get("positive_contexts") or []:
            item = item or {}; meta = item.get("metadata") or {}
            yield meta.get("law_title"), meta.get("section"), item.get("context")
    else:
        for item in row.get("relevant_laws") or []:
            item = item or {}
            yield item.get("law_name"), item.get("section_num"), item.get("section_content")

test_texts = {search_text(row.get("question")) for row in raw["test"] if search_text(row.get("question"))}
usable, audit_rows, mapping_rows = {}, [], []
for split in ("pilot", "test"):
    accepted=[]
    for index, row in enumerate(raw[split], 1):
        qid=f"{split}-{index:05d}"; question=search_text(row.get("question"))
        entries=list(gold_entries(row, split)); ids=[]; statuses=[]
        for name, section, text in entries:
            sid, status=match_statute(name, section, text)
            statuses.append(status)
            mapping_rows.append({"split":split,"question_id":qid,"law_name":name,"section":section,"status":status,"statute_id":sid})
            if sid: ids.append(sid)
        ids=list(dict.fromkeys(ids))
        overlap=split == "pilot" and bool(question) and question in test_texts
        fully_mapped=bool(entries) and len(statuses)==len(entries) and all(x.startswith("matched") for x in statuses)
        reason = "blank_question" if not question else "test_overlap" if overlap else "no_gold" if not entries else "gold_not_fully_mapped" if not fully_mapped else "included"
        audit_rows.append({"split":split,"question_id":qid,"reason":reason,"gold_entries":len(entries),"mapped_unique":len(ids)})
        if reason == "included":
            answer_field="positive_answer" if split=="pilot" else "reference_answer"
            accepted.append({"question_id":qid,"question":row["question"],"gold_ids":ids,"gold_law":corpus[ids[0]]["law_name"],"reference_answer":row.get(answer_field) or ""})
    usable[split]=pd.DataFrame(accepted, columns=["question_id","question","gold_ids","gold_law","reference_answer"])

data_audit=pd.DataFrame(audit_rows)
mapping_audit=pd.DataFrame(mapping_rows, columns=["split","question_id","law_name","section","status","statute_id"])
data_audit.to_csv(RUN_DIR / "question_audit.csv", index=False, encoding="utf-8-sig")
mapping_audit.to_csv(RUN_DIR / "mapping_audit.csv", index=False, encoding="utf-8-sig")
display(pd.crosstab(data_audit.split, data_audit.reason))
display(data_audit[data_audit.reason.eq("included")].groupby("split").gold_entries.value_counts().rename("questions").to_frame())
questions=usable[MODE].copy()
if questions.empty: raise ValueError("ไม่มีคำถามที่ใช้ได้: ตรวจ mapping_audit.csv")
if MODE == "pilot": questions=questions.sample(n=min(PILOT_QUESTIONS,len(questions)),random_state=SEED)
questions=questions.reset_index(drop=True)
print("Evaluation questions:",len(questions))

## 4. เลือกคำถามสำหรับ Generation
เลือกก่อนประเมิน retrieval ใช้คำถามชุดเดียวกันทุก retriever; เลือกกระจายตามกฎหมายโดยอาศัย gold แรกและ seed. กลุ่มกฎหมายที่เล็กอาจไม่ถูกเลือก ตรวจสัดส่วนด้วย. Pilot 4 ข้อเป็น smoke test ไม่เพียงพอสรุปผล.

In [ ]:
generation_limit = PILOT_GENERATION_QUESTIONS if MODE == "pilot" else TEST_GENERATION_QUESTIONS
eligible = questions.loc[questions["reference_answer"].str.strip().ne("")]
if eligible.empty:
    raise ValueError("ไม่มีคำตอบอ้างอิงสำหรับประเมิน Generation")

grouped = eligible.sample(frac=1, random_state=SEED).sort_values("gold_law", kind="stable")
sample_size = min(generation_limit, len(grouped))
positions = ((np.arange(sample_size) + 0.5) * len(grouped) / sample_size).astype(int)
generation_questions = grouped.iloc[positions].set_index("question_id")

print(f"Generation: {sample_size} คำถาม × 4 วิธี = {sample_size * 4} คำตอบ")

## 5. ข้อความสำหรับค้นและความยาวมาตรา
การทดลองหลักใช้ตัวบทที่ normalize จาก corpus เดียวกัน ทั้งสี่วิธี; generator ใช้ตัวบทต้นฉบับ. ยังไม่แบ่งวรรค. ตรวจความยาวด้วย BGE tokenizer และบันทึกข้อมูลที่เกิน 4096 tokens; ถ้าถูกตัด representation ของ BGE จะไม่เห็นเนื้อหาทั้งมาตรา ต้องรายงานข้อจำกัดและพิจารณา chunking เสริมบน dev.

In [ ]:
from transformers import AutoTokenizer
bge_revision = api.model_info(BGE_ID).sha
length_tokenizer = AutoTokenizer.from_pretrained(BGE_ID, revision=bge_revision)
length_rows = []
for sid, row in tqdm(corpus.items(), desc="BGE token lengths"):
    text=search_text(row["section_content"])
    length=len(length_tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])
    length_rows.append({"statute_id":sid,"chars":len(text),"bge_tokens":length,"truncated_for_bge":length>BGE_MAX_TOKENS})
length_df=pd.DataFrame(length_rows)
length_df.to_csv(RUN_DIR / "corpus_lengths.csv",index=False)
display(length_df[["chars","bge_tokens"]].describe(percentiles=[.5,.9,.95,.99]))
print("Exceed BGE limit:", int(length_df.truncated_for_bge.sum()), "/",len(length_df))
del length_tokenizer

## 6. Retrieval: ตัวแทนข้อความและ exact ranking
R1–R3 ใช้ tokenizer เดียวกัน, Word2Vec ฝึกจาก corpus ด้วย seed และ worker เดียว; R4 ใช้ tokenizer ของโมเดล. คะแนน cosine ใช้จัดอันดับภายในแต่ละวิธีเท่านั้น; tie ตามลำดับ corpus snapshot. Zero-vector query คืนผลว่างและบันทึกไว้. เวลา query รวมการ transform และ rank; การตัดคำร่วม R1–R3 รายงานแยก. Zero vector ของเอกสารต้องตรวจเพิ่มก่อนสรุปผล.

In [ ]:
def tokenize(text):
    return word_tokenize(text, engine="newmm", keep_whitespace=False)

def keep_tokens(tokens):
    return tokens

def rank_exact(query_vectors, statute_vectors):
    all_ranks = []
    for start in range(0, query_vectors.shape[0], 64):
        scores = query_vectors[start:start + 64] @ statute_vectors.T
        if issparse(scores):
            scores = scores.toarray()
        for offset, row in enumerate(scores):
            q = query_vectors[start + offset]
            is_zero = q.nnz == 0 if issparse(q) else not np.any(q)
            if is_zero:
                all_ranks.append([])
                continue
            order = np.argsort(-row, kind="stable")[:RANK_K]
            all_ranks.append([statute_ids[i] for i in order])
    return all_ranks

def word2vec_average(token_rows, model):
    vectors = np.zeros((len(token_rows), model.vector_size), dtype=np.float32)
    for i, tokens in enumerate(token_rows):
        known = [model.wv[token] for token in tokens if token in model.wv]
        if known:
            vectors[i] = np.mean(known, axis=0)
    return normalize(vectors)

statute_ids = list(corpus)
statute_texts = [search_text(corpus[sid]["section_content"]) for sid in statute_ids]
query_texts = [search_text(question) for question in questions["question"]]
start = perf_counter()
statute_tokens = [tokenize(text) for text in statute_texts]
query_tokens = [tokenize(text) for text in query_texts]
shared_tokenize_s = perf_counter() - start

ranks, costs = {}, {}
sparse_config = dict(analyzer=keep_tokens, dtype=np.float32)
for method, vectorizer in {
    "R1 Binary BoW": CountVectorizer(binary=True, **sparse_config),
    "R2 TF-IDF": TfidfVectorizer(**sparse_config),
}.items():
    start = perf_counter()
    statute_vectors = normalize(vectorizer.fit_transform(statute_tokens))
    print(method, "zero corpus vectors:", int((np.asarray(statute_vectors.getnnz(axis=1)) == 0).sum()))
    index_s = perf_counter() - start
    start = perf_counter()
    query_vectors = normalize(vectorizer.transform(query_tokens))
    ranks[method] = rank_exact(query_vectors, statute_vectors)
    costs[method] = (index_s, 1000 * (perf_counter() - start) / len(questions))

method = "R3 Word2Vec"
start = perf_counter()
w2v = Word2Vec(
    sentences=statute_tokens, sg=1, vector_size=150, window=5,
    min_count=2, epochs=10, workers=1, seed=SEED,
)
statute_vectors = word2vec_average(statute_tokens, w2v)
print("R3 zero corpus vectors:", int((np.linalg.norm(statute_vectors, axis=1)==0).sum()))
index_s = perf_counter() - start
start = perf_counter()
query_vectors = word2vec_average(query_tokens, w2v)
ranks[method] = rank_exact(query_vectors, statute_vectors)
costs[method] = (index_s, 1000 * (perf_counter() - start) / len(questions))

In [ ]:
method = "R4 BGE-M3"
encoder = SentenceTransformer(
    BGE_ID, revision=bge_revision, device=DEVICE,
    model_kwargs={"torch_dtype": DTYPE},
)
encoder.max_seq_length = BGE_MAX_TOKENS
encode_options = dict(batch_size=BGE_BATCH_SIZE, show_progress_bar=True)
start = perf_counter()
statute_vectors = normalize(encoder.encode(statute_texts, **encode_options).astype(np.float32))
index_s = perf_counter() - start

start = perf_counter()
query_vectors = normalize(encoder.encode(query_texts, **encode_options).astype(np.float32))
ranks[method] = rank_exact(query_vectors, statute_vectors)
costs[method] = (index_s, 1000 * (perf_counter() - start) / len(questions))
del encoder, statute_vectors, query_vectors
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print(f"เวลาตัดคำร่วม R1–R3: {shared_tokenize_s:.1f} วินาที")

## 7. Retrieval evaluation
รายงาน Recall@1/5/10, Hit@5, nDCG@5/10 และ MRR@10 แบบ macro เฉลี่ยรายคำถาม; relevance เป็น binary เทียบเฉพาะ gold ที่ benchmark ระบุ. เมื่อมี gold หนึ่งมาตรา Recall@k = Hit@k. Unlabeled ไม่จำเป็นต้องไม่เกี่ยวข้อง. สูตรทดสอบด้วยกรณีคำนวณมือก่อนใช้งาน.

In [ ]:
def score_ranking(gold, ranked, k):
    gold=set(gold)
    if not gold: raise ValueError("gold ต้องไม่ว่าง")
    ranked=list(dict.fromkeys(ranked))[:k]
    rel=[int(sid in gold) for sid in ranked]
    recall=sum(rel)/len(gold)
    hit=int(any(rel))
    dcg=sum(r/np.log2(i+2) for i,r in enumerate(rel))
    ideal=sum(1/np.log2(i+2) for i in range(min(k,len(gold))))
    ndcg=dcg/ideal
    first=next((i for i,r in enumerate(rel,1) if r),None)
    rr=1/first if first else 0.0
    return {"recall":recall,"hit":hit,"ndcg":ndcg,"rr":rr}

case=score_ranking(["A","B"],["X","A","Y","B","Z"],5)
assert case["recall"] == 1 and case["hit"] == 1 and case["rr"] == 0.5
assert np.isclose(case["ndcg"], (1/np.log2(3)+1/np.log2(5))/(1+1/np.log2(3)))
assert score_ranking(["A"],[],5)["ndcg"] == 0
assert score_ranking(["A"],["A","A"],5)["recall"] == 1
print("Metric checks passed")

In [ ]:
retrieval_rows=[]
for method, ranked_lists in ranks.items():
    for question, ranked in zip(questions.itertuples(index=False), ranked_lists):
        rank=next((i for i,sid in enumerate(ranked,1) if sid in question.gold_ids),0)
        row={"method":method,"question_id":question.question_id,"question":question.question,"gold_ids":question.gold_ids,"retrieved_ids":ranked,"first_gold_rank":rank,"empty_retrieval":int(not ranked)}
        for k in (1,5,10):
            result=score_ranking(question.gold_ids,ranked,k)
            for key in ("recall","hit","ndcg","rr"): row[f"{key}_at_{k}"]=result[key]
        retrieval_rows.append(row)
retrieval=pd.DataFrame(retrieval_rows)
metric_cols=["recall_at_1","recall_at_5","recall_at_10","hit_at_5","ndcg_at_5","ndcg_at_10","rr_at_10"]
def summarize_retrieval(rows):
    result=rows.groupby("method",sort=False)[metric_cols].mean().rename(columns={"rr_at_10":"mrr_at_10"})
    result["questions"]=rows.groupby("method").size()
    result["empty_queries"]=rows.groupby("method").empty_retrieval.sum()
    return result
cost_table=pd.DataFrame.from_dict(costs,orient="index",columns=["index_seconds","query_ms_per_question"])
retrieval_summary=summarize_retrieval(retrieval).join(cost_table)
generation_retrieval=retrieval.loc[retrieval.question_id.isin(generation_questions.index)].copy()
subset_summary=summarize_retrieval(generation_retrieval)
display(retrieval_summary.round(4),subset_summary.round(4))
retrieval.to_json(RUN_DIR / "retrieval_rows.jsonl",orient="records",lines=True,force_ascii=False)
retrieval_summary.to_csv(RUN_DIR / "retrieval_summary.csv")
subset_summary.to_csv(RUN_DIR / "retrieval_on_generation_subset.csv")

## 8. สร้างคำตอบและตรวจ evidence
Generator เดียวกัน, greedy decoding และ prompt เดียวกัน. เลือกข้อความเต็มมาตราในลำดับที่ค้นได้โดยไม่เกิน token budget; มาตราที่ไม่พอจะข้ามและบันทึกไว้. อัตรา gold in prompt เป็น any-hit; context recall แบบสัดส่วน gold คำนวณแยก. Generation ใช้เวลามาก; หากไม่ต้องการสร้าง ตั้ง `RUN_GENERATION=False` แล้วรันใหม่.

In [ ]:
if RUN_GENERATION:
    generator_revision = api.model_info(GENERATOR_ID).sha
    generator_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_ID, revision=generator_revision)
    generator = AutoModelForCausalLM.from_pretrained(
        GENERATOR_ID, revision=generator_revision, torch_dtype=DTYPE,
    ).to(DEVICE).eval()

    ABSTAIN_TEXT = "หลักฐานไม่เพียงพอ"
    SYSTEM_PROMPT = (
        "ตอบคำถามกฎหมายเป็นภาษาไทยโดยใช้เฉพาะตัวบทที่ให้มา "
        "สรุปคำตอบและระบุเงื่อนไขสำคัญให้ชัดเจน "
        "คัดลอกรหัสมาตราพร้อมวงเล็บเหลี่ยมจากตัวบท วางทีละรหัสหลังข้อความที่มาตรานั้นรองรับ "
        "ใช้เฉพาะรหัสที่มีในตัวบท ถ้าหลักฐานไม่พอให้ขึ้นต้นด้วย 'หลักฐานไม่เพียงพอ'"
    )

    def prompt_ids(question, blocks):
        evidence = "\n\n".join(blocks) or "(ไม่มีตัวบท)"
        return generator_tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"คำถาม\n{question}\n\nตัวบท\n{evidence}"},
            ],
            tokenize=True, add_generation_prompt=True, return_tensors="pt",
        )

    def make_prompt(question, retrieved_ids):
        blocks, included, omitted = [], [], []
        input_ids = prompt_ids(question, blocks)
        if input_ids.shape[-1] > MAX_INPUT_TOKENS:
            raise ValueError("คำถามและคำสั่งยาวเกินงบ prompt")
        for sid in retrieved_ids:
            block = f"[{sid}] {corpus[sid]['section_content']}"
            candidate = prompt_ids(question, blocks + [block])
            if candidate.shape[-1] <= MAX_INPUT_TOKENS:
                blocks.append(block)
                included.append(sid)
                input_ids = candidate
            else:
                omitted.append(sid)
        return input_ids.to(DEVICE), included, omitted

    def generate_answer(input_ids):
        start = perf_counter()
        with torch.inference_mode():
            output = generator.generate(
                input_ids, attention_mask=torch.ones_like(input_ids),
                do_sample=False, use_cache=True, max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=generator_tokenizer.eos_token_id,
            )
        new_tokens = output[0, input_ids.shape[-1]:]
        answer = generator_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return answer, len(new_tokens), perf_counter() - start

    run_config = {
        "mode": MODE, "seed": SEED, "device": DEVICE, "dtype": str(DTYPE),
        "datasets": DATASETS, "dataset_revisions": dataset_revisions,
        "bge": BGE_ID, "bge_revision": bge_revision, "bge_max_tokens": BGE_MAX_TOKENS,
        "bge_batch_size": BGE_BATCH_SIZE, "generator": GENERATOR_ID,
        "generator_revision": generator_revision, "top_k": TOP_K, "rank_k": RANK_K,
        "max_input_tokens": MAX_INPUT_TOKENS, "max_new_tokens": MAX_NEW_TOKENS,
        "system_prompt": SYSTEM_PROMPT, "generation_question_ids": generation_questions.index.tolist(),
        "versions": {p: version(p) for p in [
            "datasets", "pythainlp", "gensim", "scikit-learn",
            "sentence-transformers", "transformers", "torch",
        ]},
    }
    (RUN_DIR / "run_config.json").write_text(
        json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8",
    )

    answers = []
    with (RUN_DIR / "generation_rows.jsonl").open("w", encoding="utf-8", buffering=1) as log:
        for row in tqdm(generation_retrieval.itertuples(index=False), total=len(generation_retrieval)):
            selected = row.retrieved_ids[:TOP_K]
            input_ids, included, omitted = make_prompt(row.question, selected)
            answer, output_tokens, seconds = generate_answer(input_ids)
            cited = list(dict.fromkeys(re.findall(r"\[(S\d{5})\]", answer, flags=re.ASCII)))
            valid = [sid for sid in cited if sid in included]
            gold_in_top_k = bool(set(row.gold_ids).intersection(selected))
            gold_in_prompt = bool(set(row.gold_ids).intersection(included))
            review_key = f"{RUN_DIR.name}|{row.question_id}|{row.method}"
            review_id = "A" + sha256(review_key.encode()).hexdigest()[:12]
            record = {
                "review_id": review_id, "method": row.method, "question_id": row.question_id,
                "question": row.question, "reference_answer": generation_questions.at[row.question_id, "reference_answer"],
                "answer": answer, "gold_ids": row.gold_ids, "top_k_ids": selected,
                "prompt_ids": included, "omitted_ids": omitted, "cited_ids": cited,
                "invalid_ids": [sid for sid in cited if sid not in included],
                "valid_citations": len(valid), "total_citations": len(cited),
                "gold_cited": int(bool(set(row.gold_ids).intersection(valid))),
                "gold_in_top_k": int(gold_in_top_k), "gold_in_prompt": int(gold_in_prompt),
                "context_recall": len(set(row.gold_ids).intersection(included)) / len(set(row.gold_ids)),
                "abstained": int(answer.startswith(ABSTAIN_TEXT)),
                "input_tokens": input_ids.shape[-1], "output_tokens": output_tokens,
                "output_token_limit_reached": int(output_tokens == MAX_NEW_TOKENS),
                "generation_seconds": seconds,
            }
            answers.append(record)
            log.write(json.dumps(record, ensure_ascii=False) + "\n")

    generation = pd.DataFrame(answers)
else:
    print("Skipped generation: RUN_GENERATION=False")

## 9. สรุป citation และสร้าง blind review
Citation validity ตรวจว่ารหัสอยู่ใน prompt เท่านั้น; ไม่ยืนยันว่ารองรับคำตอบ. ผู้ประเมินให้ coverage 0–2, contradiction 0/1 และ citation_support 0/1; หากไม่มี citation ให้เว้น citation_support แล้วบันทึกเหตุผล (รายงานจำนวนที่ให้คะแนน). ตัวอย่าง pilot ที่น้อยยังสรุปเชิงวิจัยไม่ได้.

In [ ]:
if RUN_GENERATION:
    generation["has_citation"] = generation["total_citations"].gt(0)
    generation["evidence_stage"] = np.select(
        [generation["gold_in_top_k"].eq(0), generation["gold_in_prompt"].eq(0)],
        ["retrieval_miss", "context_omitted"],
        default="gold_in_prompt",
    )
    generation_summary = generation.groupby("method", sort=False).agg(
        answers=("review_id", "size"),
        citation_rate=("has_citation", "mean"),
        valid_citations=("valid_citations", "sum"),
        total_citations=("total_citations", "sum"),
        gold_citation_rate=("gold_cited", "mean"),
        gold_in_prompt_rate=("gold_in_prompt", "mean"),
        mean_context_recall=("context_recall", "mean"),
        abstention_rate=("abstained", "mean"),
        mean_input_tokens=("input_tokens", "mean"),
        mean_output_tokens=("output_tokens", "mean"),
        output_token_limit_rate=("output_token_limit_reached", "mean"),
        mean_generation_seconds=("generation_seconds", "mean"),
    )
    generation_summary["citation_validity"] = (
        generation_summary["valid_citations"]
        / generation_summary["total_citations"].replace(0, np.nan)
    )
    generation_summary.to_csv(RUN_DIR / "generation_summary.csv")

    stages = pd.crosstab(generation["method"], generation["evidence_stage"])
    stages.to_csv(RUN_DIR / "evidence_stages.csv")
    display(generation_summary.round(4), stages)

    review = generation[[
        "review_id", "question_id", "question", "reference_answer", "answer", "cited_ids", "prompt_ids",
    ]].copy()
    review["context_text"] = review["prompt_ids"].map(
        lambda ids: "\n\n".join(f"[{sid}] {corpus[sid]['section_content']}" for sid in ids)
    )
    review = review.drop(columns="prompt_ids").assign(
        coverage="", contradiction="", citation_support="", note="",
    )
    review.sample(frac=1, random_state=SEED).to_csv(
        RUN_DIR / "review_template.csv", index=False, encoding="utf-8-sig",
    )

    example = generation.loc[generation["question_id"].eq(generation_questions.index[0])].copy()
    example["มาตราที่อ้างได้"] = example.apply(
        lambda row: [
            f"{corpus[sid]['law_name']} มาตรา {corpus[sid]['section_num']}"
            for sid in row["cited_ids"] if sid in row["prompt_ids"]
        ],
        axis=1,
    )
    print("คำถามตัวอย่าง:", example.iloc[0]["question"])
    display(example[["method", "answer", "มาตราที่อ้างได้", "invalid_ids", "evidence_stage"]])
else:
    print("Skipped generation summary")

## 10. นำคะแนนคนประเมินกลับมา
ดาวน์โหลด review_template.csv กรอกคะแนนและอัปโหลดเป็น review_scored.csv ใน RUN_DIR แล้วรัน cell ต่อไป. Summary ใช้เฉพาะคำถามที่มีคะแนนครบทุกวิธีสำหรับทั้งสามช่อง; รายงานจำนวนที่ผ่านเกณฑ์และหมายเหตุกรณีไม่มี citation เพื่อหลีกเลี่ยงการตีความคลาดเคลื่อน.

In [ ]:
if RUN_GENERATION:
    score_path = RUN_DIR / "review_scored.csv"
    if not score_path.exists():
        print(f"กรอกคะแนนใน review_template.csv แล้วบันทึกเป็น {score_path} จากนั้นรันเซลล์นี้")
    else:
        score_columns = {"coverage": [0, 1, 2], "contradiction": [0, 1], "citation_support": [0, 1]}
        scores = pd.read_csv(score_path, usecols=["review_id", *score_columns])
        if scores["review_id"].duplicated().any() or not scores["review_id"].isin(generation["review_id"]).all():
            raise ValueError("review_id ซ้ำหรือไม่ตรงกับผลรันนี้")
        for column, allowed in score_columns.items():
            present = scores[column].notna()
            scores[column] = pd.to_numeric(scores[column], errors="coerce")
            if not scores.loc[present, column].isin(allowed).all():
                raise ValueError(f"คะแนน {column} ต้องอยู่ใน {allowed}")
        scored = generation.merge(scores.dropna(), on="review_id", validate="one_to_one")
        complete_questions = scored.groupby("question_id")["method"].nunique().eq(len(ranks))
        matched = scored.loc[scored["question_id"].isin(complete_questions[complete_questions].index)]
        if matched.empty:
            print("ยังไม่มีคำถามที่กรอกคะแนนครบทั้งสี่วิธี")
        else:
            human_summary = matched.groupby("method", sort=False).agg(
                reviewed_answers=("review_id", "size"),
                mean_coverage=("coverage", "mean"),
                contradiction_rate=("contradiction", "mean"),
                citation_support_rate=("citation_support", "mean"),
            ).join(generation_summary[["answers"]])
            human_summary.to_csv(RUN_DIR / "human_summary.csv")
            display(human_summary.round(4))
            display(matched.groupby(["method", "evidence_stage"])["coverage"].agg(["count", "mean"]).round(4))
else:
    print("Skipped human review")

## 11. ส่งออกผลลัพธ์
ZIP เฉพาะ RUN_DIR ปัจจุบัน. Retrieval rows และ generation rows รองรับการวิเคราะห์รายข้อ; manifest บันทึก dataset revisions และ versions. อย่าเปิดผล test เพื่อปรับค่าแล้วรายงานว่าเป็นผลทดสอบอิสระ.

In [ ]:
import shutil
archive=shutil.make_archive(str(RUN_DIR), "zip", root_dir=str(RUN_DIR))
print("Results archive:",archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("ดาวน์โหลดไฟล์ ZIP จาก filesystem ของ environment นี้")